In [ ]:
import os
import pandas as pd                                         # for data manipulation
import torch                                                # for tensor computations
from transformers import GPT2LMHeadModel, GPT2Tokenizer     # for GPT-2 model and tokenizer
from tqdm import tqdm                                       # for progress bar
import numpy as np                                          # for numerical operations
import nltk                                                 # Natural Language Toolkit
from nltk.tokenize import sent_tokenize                     # for sentence tokenization
import re                                                   # for regex operations
import spacy                                                # for NLP processing
from empath import Empath                                   # for LIWC analysis
from sentence_transformers import SentenceTransformer, util  # for semantic analysis
import collections                                          # for counting duplicates
import random                                               # for random seed setting

import gc
from transformers import AutoModel, AutoTokenizer

# Download necessary NLTK resources
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ==================== REPRODUCIBILITY SETUP ====================
SEED = 999
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"[REPRODUCIBILITY] Fixed seed set to: {SEED}")
print(f"[REPRODUCIBILITY] All random operations will use SEED={SEED}")

[REPRODUCIBILITY] Fixed seed set to: 999
[REPRODUCIBILITY] All random operations will use SEED=999


# Feature Engineering Pipeline

In [7]:

# ==================== FEATURE EXTRACTION HELPERS ====================

def is_valid_text(text):
    """Check if text is a non-empty string."""
    return isinstance(text, str) and len(text.strip()) > 0


def calculate_perplexity(text, model, tokenizer, device, stride=512):
    """Calculate perplexity using GPT-2 with a sliding window."""
    if not is_valid_text(text):
        return None

    original_max_length = tokenizer.model_max_length
    tokenizer.model_max_length = int(1e30)
    
    encodings = tokenizer(text, return_tensors="pt", truncation=False)
    tokenizer.model_max_length = original_max_length
    
    seq_len = encodings.input_ids.size(1)
    max_length = model.config.n_positions

    nlls = []
    prev_end_loc = 0

    for begin_loc in range(0, seq_len, stride):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - prev_end_loc

        input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
        target_ids = input_ids.clone()
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            with torch.amp.autocast(device_type=device):
                outputs = model(input_ids, labels=target_ids)
                neg_log_likelihood = outputs.loss * trg_len

        nlls.append(neg_log_likelihood)
        prev_end_loc = end_loc
        if end_loc == seq_len:
            break

    return torch.exp(torch.stack(nlls).sum() / seq_len).item()


def process_perplexity(texts_list, model, tokenizer, device):
    """Batch perplexity computation."""
    return [calculate_perplexity(text, model, tokenizer, device) for text in tqdm(texts_list, desc="Calculating Perplexity")]


def calculate_burstiness(text):
    """Sentence-length variance."""
    if not is_valid_text(text):
        return 0.0
    sentences = sent_tokenize(text)
    lengths = np.array([len(s.split()) for s in sentences if len(s.split()) > 0])
    if len(lengths) < 2:
        return 0.0
    return float(np.std(lengths))


def process_burstiness_list(texts_list):
    """Vectorized burstiness calculation using NumPy operations."""
    results = []
    for text in tqdm(texts_list, desc="Calculating Burstiness"):
        if is_valid_text(text):
            sentences = sent_tokenize(text)
            lengths = np.array([len(s.split()) for s in sentences if len(s.split()) > 0])
            if len(lengths) >= 2:
                results.append(float(np.std(lengths)))
            else:
                results.append(0.0)
        else:
            results.append(0.0)
    return results


def calculate_ttr(text):
    """Type-Token Ratio (lexical diversity)."""
    if not is_valid_text(text):
        return 0.0
    words = re.findall(r"\w+", text.lower())
    if not words:
        return 0.0
    return float(len(set(words)) / len(words))


def process_ttr_list(texts_list):
    """Vectorized TTR calculation using NumPy operations."""
    results = np.zeros(len(texts_list), dtype=np.float32)
    for idx, text in enumerate(tqdm(texts_list, desc="Calculating Lexical Diversity (TTR)")):
        if is_valid_text(text):
            words = re.findall(r"\w+", text.lower())
            if words:
                results[idx] = float(len(set(words)) / len(words))
    return results


def extract_stylometric_features(texts_series):
    """Vectorized extraction of stylometric features for a pandas Series."""
    complex_ratios = np.zeros(len(texts_series), dtype=np.float32)
    avg_word_lens = np.zeros(len(texts_series), dtype=np.float32)
    avg_sent_lens = np.zeros(len(texts_series), dtype=np.float32)
    
    for idx, text in enumerate(tqdm(texts_series, desc="Extracting Stylometric Features")):
        if is_valid_text(text):
            words = re.findall(r"\w+", text.lower())
            sentences = sent_tokenize(text)
            if words and sentences:
                avg_word_lens[idx] = float(np.mean([len(w) for w in words]))
                avg_sent_lens[idx] = float(len(words) / len(sentences))
                complex_ratios[idx] = float(len([w for w in words if len(w) > 6]) / len(words))
    
    return complex_ratios, avg_word_lens, avg_sent_lens


def calculate_uid(text, model, tokenizer, device, max_length=1024, stride=512):
    """Uniform Information Density (std of surprisal) with sliding window for long texts."""
    if not is_valid_text(text):
        return 0.0
    
    # Temporarily increase model_max_length to suppress warnings
    original_max_length = tokenizer.model_max_length
    tokenizer.model_max_length = int(1e30)
    
    inputs = tokenizer(text, return_tensors="pt", truncation=False)
    tokenizer.model_max_length = original_max_length
    
    input_ids = inputs["input_ids"]
    seq_len = input_ids.size(1)
    
    if seq_len < 2:
        return 0.0
    
    all_surprisals = []
    
    # Sliding window for long texts
    for begin_loc in range(0, seq_len, stride):
        end_loc = min(begin_loc + max_length, seq_len)
        
        chunk_ids = input_ids[:, begin_loc:end_loc].to(device)
        
        if chunk_ids.shape[1] < 2:
            continue
            
        with torch.no_grad():
            with torch.amp.autocast(device_type=device, enabled=(device == "cuda")):
                outputs = model(chunk_ids, labels=chunk_ids)
                logits = outputs.logits
        
        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = chunk_ids[:, 1:].contiguous()
        loss_fct = torch.nn.CrossEntropyLoss(reduction="none")
        surprisal = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
        all_surprisals.append(surprisal)
        
        if end_loc >= seq_len:
            break
    
    if not all_surprisals:
        return 0.0
    
    # Concatenate all surprisals and compute std - VECTORIZED
    combined_surprisals = torch.cat(all_surprisals)
    return float(torch.std(combined_surprisals).item())


def extract_empath(df, text_column, lexicon, categories=None):
    """Vectorized Empath LIWC features using batch processing."""
    print(f"Processing Empath features for: {text_column} (categories: {categories})...")
    feats = []
    
    # Batch process texts for efficiency
    batch_size = 32
    texts = df[text_column].tolist()
    
    for i in tqdm(range(0, len(texts), batch_size), desc=f"{text_column} - Empath"):
        batch_texts = texts[i:i+batch_size]
        for text in batch_texts:
            if is_valid_text(text):
                res = lexicon.analyze(text, categories=categories, normalize=True) or {}
            else:
                res = {}
            res_filled = {cat: res.get(cat, 0.0) for cat in categories}
            feats.append(res_filled)
    
    temp_df = pd.DataFrame(feats).add_prefix(f"{text_column}_")
    return temp_df


def calculate_avg_syntax_depth(text, nlp):
    """Average parse-tree depth per sentence."""
    if not is_valid_text(text):
        return 0.0
    doc = nlp(text)
    def walk(node, depth):
        if node.n_lefts + node.n_rights == 0:
            return depth
        return max(walk(child, depth + 1) for child in node.children)
    depths = [walk(sent.root, 1) for sent in doc.sents]
    return float(np.mean(depths)) if depths else 0.0

    
def get_semantic_features_batch(texts_list, semantic_model, device):
    """Vectorized semantic feature extraction using batch encoding."""
    results = []
    
    valid_texts = [(i, text) for i, text in enumerate(texts_list) if is_valid_text(text)]
    
    if not valid_texts:
        return np.zeros((len(texts_list), 2), dtype=np.float32)
    
    all_means = np.zeros(len(texts_list), dtype=np.float32)
    all_stds = np.zeros(len(texts_list), dtype=np.float32)
    
    for idx, text in tqdm(valid_texts, desc="Semantic Features", total=len(valid_texts)):
        sentences = [s for s in sent_tokenize(text) if len(s.split()) > 1]
        if len(sentences) >= 2:
            with torch.no_grad():
                embeddings = semantic_model.encode(sentences, convert_to_tensor=True, device=device)
            
            sims = torch.tensor([util.cos_sim(embeddings[i], embeddings[i+1]).item() 
                                 for i in range(len(embeddings) - 1)])
            all_means[idx] = float(torch.mean(sims).item())
            
            if len(sims) > 1:
                all_stds[idx] = float(torch.std(sims).item())
            else:
                all_stds[idx] = 0.0
    
    return all_means, all_stds

# BERT

In [8]:
# ==================== BERT CONFIGURATION ====================

MODEL_NAME = "bert-base-uncased"
BATCH_SIZE = 8
MAX_LENGTH = 512
CHUNK_OVERLAP = 50

# Global model and tokenizer for BERT (loaded once)
bert_tokenizer = None
bert_model = None
bert_device = None

print(f"[CONFIG] BERT Model: {MODEL_NAME} (768-dim embeddings)")
print(f"[CONFIG] Max Length: {MAX_LENGTH} tokens, Overlap: {CHUNK_OVERLAP} tokens, Batch Size: {BATCH_SIZE}")

def initialize_bert_model(model_name=MODEL_NAME):
    
    """Initialize BERT model and tokenizer once."""
    global bert_tokenizer, bert_model, bert_device
    
    if bert_model is None:
        print(f"\nLoading BERT model: {model_name}")
        bert_tokenizer = AutoTokenizer.from_pretrained(model_name)
        bert_model = AutoModel.from_pretrained(model_name)
        
        # Freeze model weights
        for param in bert_model.parameters():
            param.requires_grad = False
        
        bert_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        bert_model.to(bert_device)
        bert_model.eval()
        print(f"Model loaded on device: {bert_device}")


def chunk_text_by_tokens(text, max_length=MAX_LENGTH, overlap=CHUNK_OVERLAP):
    """
    Split long text into overlapping chunks based on token count.
    Ensures chunks never exceed max_length tokens.
    Returns list of token tensors.
    """
    encoded = bert_tokenizer.encode_plus(
        text,
        add_special_tokens=False,
        truncation=False,
        return_tensors=None
    )
    token_ids = encoded['input_ids']
    
    # If text fits in one chunk, return it
    if len(token_ids) <= max_length - 2:  # -2 because of special tokens
        inputs = bert_tokenizer.encode_plus(
            text,
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return [inputs]
    
    # Split tokens into overlapping chunks
    chunks = []
    chunk_size = max_length - 2
    stride = chunk_size - overlap
    
    for i in range(0, len(token_ids), stride):
        chunk_ids = token_ids[i:i + chunk_size]
        
        # Convert back to text
        chunk_text = bert_tokenizer.decode(chunk_ids, skip_special_tokens=True)
        
        chunk_input = bert_tokenizer.encode_plus(
            chunk_text,
            max_length=max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        chunks.append(chunk_input)
        
        if i + chunk_size >= len(token_ids):
            break
    
    return chunks if chunks else [bert_tokenizer.encode_plus(
        text,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )]


def get_bert_embeddings(text_list, batch_size=BATCH_SIZE):
    """
    Extract BERT embeddings for long texts using token-based chunking.
    
    Args:
        text_list: List of text strings (can be very long)
        batch_size: Number of chunks to process at once    
    Returns:
        numpy array of shape (n_texts, 768) containing embeddings
    """
    all_embeddings = []
    
    for text in tqdm(text_list, desc="Extracting BERT Embeddings"):
        try:
            chunks = chunk_text_by_tokens(str(text))
            chunk_embeddings = []
            
            for i in range(0, len(chunks), batch_size):
                batch_chunks = chunks[i:i+batch_size]
                
                # Stack inputs safely
                input_ids = torch.cat([c['input_ids'] for c in batch_chunks]).to(bert_device)
                attention_mask = torch.cat([c['attention_mask'] for c in batch_chunks]).to(bert_device)
                
                with torch.no_grad():
                    outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
                    cls_embs = outputs.last_hidden_state[:, 0, :].cpu().numpy()
                    chunk_embeddings.append(cls_embs)
            
            all_chunk_embs = np.vstack(chunk_embeddings)
            text_embedding = np.mean(all_chunk_embs, axis=0)
            all_embeddings.append(text_embedding)
            
        except Exception as e:
            print(f"Error processing text: {str(e)[:100]}") 
            all_embeddings.append(np.zeros(768))  # 768 is BERT hidden size
    
    return np.array(all_embeddings)


def add_bert_features(df, output_dir,text_column='Text'):
    """
    Add BERT embeddings to a DataFrame.
    
    Args:
        df: Input DataFrame
        text_column: Name of the column containing text (default: 'Text')
    
    Returns:
        DataFrame with BERT features added
    """
    # Initialize model on first call
    initialize_bert_model()
    
    texts = df[text_column].tolist()
    
    print(f"\n{'='*60}")
    print("Adding BERT embeddings...")
    print(f"{'='*60}")
    
    bert_embeddings = get_bert_embeddings(texts)
    
    bert_columns = [f'bert_{i}' for i in range(bert_embeddings.shape[1])]
    df_bert = pd.DataFrame(bert_embeddings, columns=bert_columns)
    df_combined = pd.concat([df.reset_index(drop=True), df_bert], axis=1)
    
    print(f"\nBERT embeddings added - shape: {df_combined.shape}")
    
    # Cleanup
    del bert_embeddings, df_bert
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    print("BERT features added!")
    
    output_path = os.path.join(output_dir, "DB_final_with_BERT.csv")
    df_combined.to_csv(output_path, index=False)

[CONFIG] BERT Model: bert-base-uncased (768-dim embeddings)
[CONFIG] Max Length: 512 tokens, Overlap: 50 tokens, Batch Size: 8


# Main function to process the dataset

In [ ]:
# ========================================
# MAIN PIPELINE
# ========================================

def process_dataset(input_dataset: str, output_file: str, human_col_name: str = 'Human') -> pd.DataFrame:
    """Run full feature pipeline for the provided Writer and save everything to a single CSV."""
    
    output_dir = os.path.dirname(output_file)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    checkpoint_file = output_file.replace('.csv', '_checkpoint.csv') if output_file.endswith('.csv') else output_file + '_checkpoint.csv'

    np.random.seed(999)
    torch.manual_seed(999)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(999)

    if os.path.exists(checkpoint_file):
        print(f"[RESUME] Found checkpoint file. Loading and resuming from last step...")
        df = pd.read_csv(checkpoint_file)
        
        if 'bert_0' in df.columns:
            print("[RESUME] All steps completed! Final file exists, proceeding to save...")
            df.to_csv(output_file, index=False)
            print(f"Final dataset saved to: {output_file}")
            return df
        elif 'Text' in df.columns and 'is_AI' in df.columns:
            print("[RESUME] Resuming from step [11/11] - BERT Embeddings...")
            train_df = df.copy()
            goto_bert = True
        elif 'semantic_std' in df.columns:
            print("[RESUME] Resuming from step [10/11] - Formatting dataset...")
            goto_formatting = True
        elif 'semantic_mean' in df.columns:
            print("[RESUME] Resuming from step [9/11] - Semantic Consistency (already done)...")
            goto_semantic = False
        elif 'syntax_depth' in df.columns:
            print("[RESUME] Resuming from step [9/11] - Semantic Consistency...")
            goto_semantic = True
        elif 'Art' in [c for c in df.columns if 'Art' in c] or 'beauty' in df.columns:
            print("[RESUME] Resuming from step [8/11] - Syntax Tree Depth...")
            goto_syntax = True
        else:
            print("[RESUME] Resuming from an earlier step...")
            goto_syntax = True
            
        device = "cuda" if torch.cuda.is_available() else "cpu"
    else:
        print("[1/11] Loading and Cleaning dataset...")
        df = pd.read_csv(input_dataset)
        
        rename_dict = {}
        if 'Unnamed: 0' in df.columns: rename_dict['Unnamed: 0'] = 'Writer'
        if 'Unnamed: 1' in df.columns: rename_dict['Unnamed: 1'] = 'Article'
        if 'Writers' in df.columns: rename_dict['Writers'] = 'Writer'
        
        if rename_dict:
            df.rename(columns=rename_dict, inplace=True)
            
        if 'Writer' not in df.columns or 'Article' not in df.columns:
            raise ValueError(f"Dataset must contain 'Writer' and 'Article' columns. Found: {df.columns.tolist()}")

        CLEAN_MAPPING = {
            'Human_story': 'Human', 'Human_story_paraphrased': 'Human', 'human': 'Human', 'Human_story_translated': 'Human',
            'GPT_4-o': 'GPT-4o', 'GPT_4-o_paraphrased': 'GPT-4o', 'GPT_4-o_translated': 'GPT-4o',
            'gemma-2-9b': 'Gemma-2-9B', 'gemma-2-9b_paraphrased': 'Gemma-2-9B', 'gemma-2-9b_translated': 'Gemma-2-9B',
            'mistral-7B': 'Mistral-7B', 'mistral-7B_paraphrased': 'Mistral-7B', 'mistral-7B_translated': 'Mistral-7B',
            'llama-8B': 'Llama-8B', 'llama-8B_paraphrased': 'Llama-8B', 'llama-8B_translated': 'Llama-8B',
            'qwen-2-72B': 'Qwen-2-72B', 'qwen-2-72B_paraphrased': 'Qwen-2-72B', 'qwen-2-72B_translated': 'Qwen-2-72B',
            'accounts/yi-01-ai/models/yi-large': 'Yi-Large', 'accounts/yi-01-ai/models/yi-large_paraphrased': 'Yi-Large', 'yi-large_translated': 'Yi-Large',
            'GPT4All': 'GPT4All', 'Claude': 'Claude'
        }
        df['Writer'] = df['Writer'].map(CLEAN_MAPPING).fillna(df['Writer'])
        # -----------------------------------------------------

        Writer = df["Writer"].unique().tolist()
        print(f"Dataset shape after filtering: {df.shape}")
        print(f"Writer: {Writer}")

        device = "cuda" if torch.cuda.is_available() else "cpu"
        model_id = 'gpt2'
        gpt_tokenizer = GPT2Tokenizer.from_pretrained(model_id)
        gpt_model = GPT2LMHeadModel.from_pretrained(model_id).to(device)
        gpt_model = gpt_model.half() if device == "cuda" else gpt_model
        gpt_model.eval()

        print("[2/11] Calculating Perplexity...")
        df['ppl'] = process_perplexity(df['Article'].tolist(), gpt_model, gpt_tokenizer, device)
        df.to_csv(checkpoint_file, index=False)

        print("[3/11] Calculating Burstiness...")
        df['burstiness'] = process_burstiness_list(df['Article'].tolist())
        df.to_csv(checkpoint_file, index=False)

        print("[4/11] Calculating Type-Token Ratio (TTR)...")
        df['ttr'] = process_ttr_list(df['Article'].tolist())
        df.to_csv(checkpoint_file, index=False)

        print("[5/11] Calculating Stylometric Features...")
        complex_ratios, avg_word_lens, avg_sent_lens = extract_stylometric_features(df['Article'])
        df['complex'] = complex_ratios
        df['avg_word_len'] = avg_word_lens
        df['avg_sent_len'] = avg_sent_lens
        df.to_csv(checkpoint_file, index=False)

        print("[6/11] Calculating Uniform Information Density (UID)...")
        df['uid'] = [calculate_uid(x, gpt_model, gpt_tokenizer, device) for x in tqdm(df['Article'], desc="Calculating UID")]
        df.to_csv(checkpoint_file, index=False)
        
        del gpt_model, gpt_tokenizer
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print("[7/11] Calculating LIWC (Empath) Features...")
        USED_EMPATH_CATEGORIES = [
            'gain', 'beauty', 'government', 'urban', 'art',
            'help', 'optimism', 'strength', 'love', 'traveling',
        ]
        lexicon = Empath()
        empath_frames = extract_empath(df, 'Article', lexicon, categories=USED_EMPATH_CATEGORIES)
        empath_frames.columns = [col.replace('Article_', '') for col in empath_frames.columns]
        df = pd.concat([df, empath_frames], axis=1)
        df.to_csv(checkpoint_file, index=False)

        goto_syntax = True

    if 'goto_syntax' in locals() and goto_syntax or ('syntax_depth' not in df.columns):
        print("[8/11] Calculating Syntax Tree Depth...")
        spacy.prefer_gpu()
        nlp = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])
        
        syntax_depths = []
        texts_list = df['Article'].tolist()
        
        def get_syntax_depth(doc):
            def walk(node, depth):
                if node.n_lefts + node.n_rights == 0:
                    return depth
                return max(walk(child, depth + 1) for child in node.children)
            depths = [walk(sent.root, 1) for sent in doc.sents]
            return float(np.mean(depths)) if depths else 0.0
        
        for text in tqdm(texts_list, desc="Syntax Depth"):
            if is_valid_text(text):
                doc = nlp(text)
                syntax_depths.append(get_syntax_depth(doc))
            else:
                syntax_depths.append(0.0)
        
        df['syntax_depth'] = syntax_depths
        df.to_csv(checkpoint_file, index=False)

    if 'semantic_mean' not in df.columns:
        print("[9/11] Calculating Semantic Consistency...")
        device = "cuda" if torch.cuda.is_available() else "cpu"
        semantic_model = SentenceTransformer('all-MiniLM-L6-v2').to(device)
        means, stds = get_semantic_features_batch(df['Article'].tolist(), semantic_model, device)
        df['semantic_mean'] = means
        df['semantic_std'] = stds
        df.to_csv(checkpoint_file, index=False)

        del semantic_model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    else:
        print("[9/11] Semantic Consistency (already completed, skipping)...")

    if 'is_AI' not in df.columns:
        print("[10/11] Formatting dataset...")
        
        df = df.rename(columns={'Article': 'Text', 'Writer': 'Writer'})
        df['is_AI'] = (df['Writer'].str.lower() != human_col_name.lower()).astype(int)

        train_df = df.copy()
        print(f"Rows before dropna: {len(train_df)}")
        train_df.dropna(inplace=True)
        train_df.reset_index(drop=True, inplace=True)
        print(f"Rows after dropna: {len(train_df)}")
        train_df.to_csv(checkpoint_file, index=False)
    else:
        print("[10/11] Formatting (already completed, skipping)...")
        train_df = df.copy()

    if 'bert_0' not in df.columns:
        print("[11/11] Extracting BERT Embeddings...")
        initialize_bert_model()
        bert_embeddings = get_bert_embeddings(train_df['Text'].tolist())
        
        bert_columns = [f'bert_{i}' for i in range(bert_embeddings.shape[1])]
        df_bert = pd.DataFrame(bert_embeddings, columns=bert_columns, index=train_df.index)
        train_df = pd.concat([train_df, df_bert], axis=1)
        
        del bert_embeddings, df_bert
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    else:
        print("[11/11] BERT Embeddings (already completed, skipping)...")

    print("[DONE] Saving final unified dataset...")
    train_df.to_csv(output_file, index=False)

    if os.path.exists(checkpoint_file):
        os.remove(checkpoint_file)
        print(f"Removed temporary checkpoint file.")

    print("" + "=" * 80)
    print("PIPELINE COMPLETE!")
    print("=" * 80)
    print(f"Final DataFrame Shape: {train_df.shape}")
    print(f"Total Samples: {len(train_df)}")
    print(f"Dataset successfully saved to: {output_file}")
    print("=" * 80)
    return train_df

In [10]:
input_dir=os.path.join(os.getcwd(), "data/need_processing")
output_dir = os.path.join(os.getcwd(), "data/processed")
for csv in os.listdir(input_dir)[::-1]:
    if csv.endswith(".csv"):
        input = os.path.join(input_dir, csv)
        output = os.path.join(output_dir, f"processed_{csv}")
        human_col_name = "Human"
        process_dataset(input_dataset=input, output_file=output, human_col_name=human_col_name)

[1/11] Loading and Cleaning dataset...
Dataset shape after filtering: (2094, 2)
Writer: ['GPT-4o', 'Human', 'Yi-Large', 'Gemma-2-9B', 'Llama-8B', 'Mistral-7B', 'Qwen-2-72B']
[2/11] Calculating Perplexity...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.
Calculating Perplexity: 100%|██████████| 2094/2094 [00:33<00:00, 62.91it/s]


[3/11] Calculating Burstiness...


Calculating Burstiness: 100%|██████████| 2094/2094 [00:00<00:00, 3964.04it/s]


[4/11] Calculating Type-Token Ratio (TTR)...


...

[9/11] Calculating Semantic Consistency...


Semantic Features: 100%|██████████| 51181/51181 [57:16<00:00, 14.89it/s]


[10/11] Formatting dataset...
Rows before dropna: 51247
Rows after dropna: 51181
[11/11] Extracting BERT Embeddings...


Extracting BERT Embeddings: 100%|██████████| 51181/51181 [1:05:28<00:00, 13.03it/s]


[DONE] Saving final unified dataset...
Removed temporary checkpoint file.
PIPELINE COMPLETE!
Final DataFrame Shape: (51181, 791)
Total Samples: 51181
Dataset successfully saved to: /home/basel/Work/python/ai_text_detection/data/processed/processed_articles.csv


In [ ]:
# ==========================================
# Renaming models
# ==========================================

import os
import pandas as pd

directories = ["data/processed/", "data/need_processing/"]

CLEAN_MAPPING = {
    'Human_story': 'Human',
    'Human_story_paraphrased': 'Human',
    'human': 'Human',

    # GPT-4o
    'GPT_4-o': 'GPT-4o',
    'GPT_4-o_paraphrased': 'GPT-4o',

    # Gemma
    'gemma-2-9b': 'Gemma-2-9B',
    'gemma-2-9b_paraphrased': 'Gemma-2-9B',

    # Mistral
    'mistral-7B': 'Mistral-7B',
    'mistral-7B_paraphrased': 'Mistral-7B',

    # Llama
    'llama-8B': 'Llama-8B',
    'llama-8B_paraphrased': 'Llama-8B',

    # Qwen
    'qwen-2-72B': 'Qwen-2-72B',
    'qwen-2-72B_paraphrased': 'Qwen-2-72B',

    # Yi-Large
    'accounts/yi-01-ai/models/yi-large': 'Yi-Large',
    'accounts/yi-01-ai/models/yi-large_paraphrased': 'Yi-Large'
}

print("=" * 80)
print("STANDARDIZING ALL WRITER NAMES AND COLUMNS")
print("=" * 80)

for directory in directories:
    if not os.path.exists(directory):
        continue
        
    for file in os.listdir(directory):
        if not file.endswith(".csv"):
            continue

        file_path = os.path.join(directory, file)
        df = pd.read_csv(file_path)

        if 'Writers' in df.columns:
            df = df.rename(columns={'Writers': 'Writer'})

        if 'Writer' in df.columns:
            df['Writer'] = df['Writer'].map(CLEAN_MAPPING).fillna(df['Writer'])

        df.to_csv(file_path, index=False)
        print(f"✓ Processed & Cleaned: {directory}{file}")

print("\n" + "=" * 80)
print("VERIFICATION COMPLETE - ALL FILES STANDARDIZED")
print("=" * 80)

STANDARDIZING ALL WRITER NAMES AND COLUMNS
✓ Processed & Cleaned: data/processed/processed_cross_domain_wp.csv
✓ Processed & Cleaned: data/processed/processed_translation.csv
✓ Processed & Cleaned: data/processed/processed_unseen_reuters.csv
✓ Processed & Cleaned: data/processed/processed_cross_domain_essay.csv
✓ Processed & Cleaned: data/processed/processed_paraphrased_articles.csv
✓ Processed & Cleaned: data/processed/processed_misspelled_20.csv
✓ Processed & Cleaned: data/processed/processed_misspelled_15.csv
✓ Processed & Cleaned: data/processed/processed_misspelled_10.csv
✓ Processed & Cleaned: data/processed/processed_misspelled_5.csv
✓ Processed & Cleaned: data/processed/processed_articles.csv
✓ Processed & Cleaned: data/need_processing/articles.csv
✓ Processed & Cleaned: data/need_processing/misspelled_5.csv
✓ Processed & Cleaned: data/need_processing/misspelled_10.csv
✓ Processed & Cleaned: data/need_processing/misspelled_15.csv
✓ Processed & Cleaned: data/need_processing/miss

In [ ]:
import pandas as pd
import os
import re
import numpy as np
import nltk
from nltk.tokenize import sent_tokenize
from empath import Empath
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# ==========================================
# 1. Fixing is_AI Column
# ==========================================
print("--- 1. Fixing is_AI Column ---")
main_file_path = 'data/processed/processed_articles.csv'

if os.path.exists(main_file_path):
    df_main = pd.read_csv(main_file_path)
    if 'is_AI' not in df_main.columns:
        df_main['is_AI'] = (df_main['Writer'].str.lower() != 'human').astype(int)
        df_main.to_csv(main_file_path, index=False)
        print("is_AI column successfully added to processed_articles.csv")
    else:
        print("is_AI column already exists.")
else:
    print("processed_articles.csv not found!")

# ==========================================
# 2. (Schema Mismatch)
# ==========================================
print("\n--- 2. Fixing Schema Mismatch (Standalone Extraction) ---")

lexicon = Empath()
USED_EMPATH_CATEGORIES = [
    'gain', 'beauty', 'government', 'urban', 'art', 
    'help', 'optimism', 'strength', 'love', 'traveling'
]

def extract_missing_features(df):
    texts = df['Text'].astype(str).tolist()
    
    if 'complex' not in df.columns or 'avg_word_len' not in df.columns or 'avg_sent_len' not in df.columns:
        print("    Extracting Stylometric Features...")
        complex_ratios, avg_word_lens, avg_sent_lens = [], [], []
        
        for text in tqdm(texts, desc="Stylometric"):
            words = re.findall(r"\w+", text.lower())
            sentences = sent_tokenize(text)
            
            if words and sentences:
                avg_word_lens.append(float(np.mean([len(w) for w in words])))
                avg_sent_lens.append(float(len(words) / len(sentences)))
                complex_ratios.append(float(len([w for w in words if len(w) > 6]) / len(words)))
            else:
                avg_word_lens.append(0.0)
                avg_sent_lens.append(0.0)
                complex_ratios.append(0.0)
                
        df['complex'] = complex_ratios
        df['avg_word_len'] = avg_word_lens
        df['avg_sent_len'] = avg_sent_lens

    missing_empath = [cat for cat in USED_EMPATH_CATEGORIES if cat not in df.columns]
    if missing_empath:
        print("    Extracting Empath Features...")
        empath_results = []
        for text in tqdm(texts, desc="Empath"):
            res = lexicon.analyze(text, categories=USED_EMPATH_CATEGORIES, normalize=True)
            if not res:
                res = {c: 0.0 for c in USED_EMPATH_CATEGORIES}
            empath_results.append(res)
            
        empath_df = pd.DataFrame(empath_results)
        for col in missing_empath:
            df[col] = empath_df[col]

    return df

files_to_fix = [
    "data/processed/processed_translation.csv",
    "data/processed/processed_cross_domain_essay.csv",
    "data/processed/processed_cross_domain_wp.csv",
    "data/processed/processed_unseen_reuters.csv"
]

for file_path in files_to_fix:
    if os.path.exists(file_path):
        print(f"Processing {os.path.basename(file_path)}...")
        df_fix = pd.read_csv(file_path)
        
        df_fix = extract_missing_features(df_fix)
        
        df_fix.to_csv(file_path, index=False)
        print(f"Saved {os.path.basename(file_path)} with all features.")
    else:
        print(f"File {file_path} not found.")

# ==========================================
# 3.  (Balancing Datasets)
# ==========================================
print("\n--- 3. Balancing Datasets ---")

def balance_dataset(file_path, target_samples, seed=999):
    if not os.path.exists(file_path):
        print(f"File {file_path} not found!")
        return

    df = pd.read_csv(file_path)
    
    class_counts = df['Writer'].value_counts()
    if class_counts.min() < target_samples:
        print(f"Cannot balance {os.path.basename(file_path)}. Minimum class has {class_counts.min()}, target is {target_samples}.")
        return

    df_balanced = df.groupby('Writer').sample(n=target_samples, random_state=seed)
    df_balanced = df_balanced.sample(frac=1, random_state=seed).reset_index(drop=True)
    
    df_balanced.to_csv(file_path, index=False)
    print(f"File {os.path.basename(file_path)} balanced to {target_samples} samples per class.")

balance_dataset('data/processed/processed_paraphrased_articles.csv', 296)
balance_dataset('data/processed/processed_translation.csv', 247)

print("\n--- Process Completed ---")